# HYRAS projection definitions

HYRAS on the full domain ``x_ll=3806000.0 y_ll=2489000.0 x_ur=5096000.0 y_ur=3589000.0``:
* HYR-LAEA-1
* HYR-LAEA-5
* HYR-LAEA-12.5
* HYR-LAEA-50
* HYR-LAEA-100

HYRAS over the german domain ``x_ll=4021000.0 y_ll=2674000.0 x_ur=4686000.0 y_ur=3564000.0``:
* HYR-GER-LAEA-1
* HYR-GER-LAEA-5
* HYR-GER-LAEA-12.5
* HYR-GER-LAEA-50
* HYR-GER-LAEA-100

In [ ]:
import xarray as xr
import pyku
from pyresample import utils
from pyresample import get_area_def
import warnings

# Testing

The following commented lines permit to read exemplary data for comparison

In [ ]:
# ds = xr.open_dataset('/path/to/hyras/output/tas/v6-1/05/tas_hyras_5_1980_v6-1.nc')  # full domain
# ds = xr.open_dataset('/path/to/hyras/output/tas/v6-1/05_abgabe_de/tas_hyras_5_1980_v6-1_de.nc')  # german domain

# x_ll = ds.x.values[0] - (ds.x.values[1]-ds.x.values[0])/2.
# y_ll = ds.y.values[0] - (ds.y.values[1]-ds.y.values[0])/2.
# x_ur = ds.x.values[-1] + (ds.x.values[-1]-ds.x.values[-2])/2.
# y_ur = ds.y.values[-1] + (ds.y.values[-1]-ds.y.values[-2])/2.
# print(f"{x_ll=} {y_ll=} {x_ur=} {y_ur=}")
# print(ds.crs.attrs['proj4'])
# ds.crs.attrs

# Define proj4 parameters

In [ ]:
proj4_args = '+proj=laea +lat_0=52 +lon_0=10 +x_0=4321000 +y_0=3210000 +ellps=GRS80 +towgs84=0,0,0,0,0,0,0 +units=m +no_defs +type=crs'

# Define area extent for the full HYRAS domain and for Germany

In [ ]:
x_ll=3806000.0
y_ll=2489000.0
x_ur=5096000.0
y_ur=3589000.0

area_extent_full = (x_ll, y_ll, x_ur, y_ur)

x_ll=4021000.0
y_ll=2674000.0
x_ur=4686000.0
y_ur=3564000.0
area_extent_germany = (x_ll, y_ll, x_ur, y_ur)

print(f"{area_extent_full=}")
print(f"{area_extent_germany=}")

# Define function to generate pyresample area definition

In [ ]:
def generate_area_def(area_extent, resolution_km=1, proj_identifier='HYR-LAEA'):

    x_ll, y_ll, x_ur, y_ur = area_extent
    area_id = f'{proj_identifier}-{resolution_km}'
    area_name = f'{proj_identifier}-{resolution_km}'
    
    proj_id = None
    
    resolution_m = resolution_km*1000.0

    # Determine the y and x size, and throw a warning if not an integer
    
    def calculate_size(coord_ur, coord_ll, res):
        raw_value = (coord_ur - coord_ll) / res
        
        if raw_value % 1 != 0:
            warnings.warn(f"{proj_identifier}-{resolution_km} Calculation result {raw_value} has decimals and will be rounded.")
        
        return int(round(raw_value, 0))

    x_size = calculate_size(x_ur, x_ll, resolution_m)
    y_size = calculate_size(y_ur, y_ll, resolution_m)
    
    area_extent = (x_ll, y_ll, x_ur, y_ur)
    area_def = get_area_def(
        area_id,
        area_name,
        proj_id,
        proj4_args,
        x_size,
        y_size,
        area_extent
    )
    
    print(area_def.dump())

# Define function to generate crs metadata

In [ ]:
def generate_metadata(area_extent, resolution_km=1, proj_identifier='HYR-LAEA'):

    x_ll, y_ll, x_ur, y_ur = area_extent
    resolution_m = resolution_km*1000.0

    # Determine the y and x size, and throw a warning if not an integer
    
    def calculate_size(coord_ur, coord_ll, res):
        raw_value = (coord_ur - coord_ll) / res
        
        if raw_value % 1 != 0:
            warnings.warn(f"{proj_identifier}-{resolution_km} Calculation result {raw_value} has decimals and will be rounded.")
        
        return int(round(raw_value, 0))

    x_size = calculate_size(x_ur, x_ll, resolution_m)
    y_size = calculate_size(y_ur, y_ll, resolution_m)
   
    print(
f"""{proj_identifier}-{resolution_km}:
    crs_name: crs
    crs_data:
        grid_mapping_name: 'lambert_azimuthal_equal_area'
        longitude_of_central_meridian: 10.0
        latitude_of_projection_origin: 52.0
        semi_major_axis: 6378137.0
        semi_minor_axis: 6356752.31414036
        inverse_flattening: 298.257222101
        false_easting: 4321000.0
        false_northing: 3210000.0
        spatial_ref: 'PROJCS["ETRS_1989_LAEA",GEOGCS["GCS_ETRS_1989",DATUM["D_ETRS_1989",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["False_Easting",4321000.0],PARAMETER["False_Northing",3210000.0],PARAMETER["Central_Meridian",10.0],PARAMETER["Latitude_Of_Origin",52.0],UNIT["Meter",1.0]]'
        crs_wkt: 'PROJCS["ETRS_1989_LAEA",GEOGCS["GCS_ETRS_1989",DATUM["D_ETRS_1989",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["False_Easting",4321000.0],PARAMETER["False_Northing",3210000.0],PARAMETER["Central_Meridian",10.0],PARAMETER["Latitude_Of_Origin",52.0],UNIT["Meter",1.0]]'
        proj4: '+proj=laea +lat_0=52 +lon_0=10 +x_0=4321000 +y_0=3210000 +ellps=GRS80 +towgs84=0,0,0,0,0,0,0 +units=m +no_defs +type=crs'
        proj4_params: '+proj=laea +lat_0=52 +lon_0=10 +x_0=4321000 +y_0=3210000 +ellps=GRS80 +towgs84=0,0,0,0,0,0,0 +units=m +no_defs +type=crs'
        epsg_code: 'EPSG:3035'
        long_name: 'DWD HYRAS ETRS89 LAEA grid with {x_size} columns and {y_size} rows'
""")

# Generate pyresample projections

In [ ]:
# HYRAS full extent
for resolution_km in [1, 2, 3, 5, 12.5, 50, 100]:
    generate_area_def(area_extent = area_extent_full, resolution_km=resolution_km, proj_identifier='HYR-LAEA')

# HYRAS Germany
for resolution_km in [1, 2, 3, 5, 12.5, 50, 100]:
    generate_area_def(area_extent = area_extent_germany, resolution_km=resolution_km, proj_identifier='HYR-GER-LAEA')

# Generate crs metadata

In [ ]:
# HYRAS full extent
for resolution_km in [1, 2, 3, 5, 12.5, 50, 100]:
    generate_metadata(area_extent = area_extent_full, resolution_km=resolution_km, proj_identifier='HYR-LAEA')

# HYRAS Germany
for resolution_km in [1, 2, 3, 5, 12.5, 50, 100]:
    generate_metadata(area_extent = area_extent_germany, resolution_km=resolution_km, proj_identifier='HYR-GER-LAEA')